# Day 5 · Concept 4 of 5: Do Not Flood the Tool

**Big idea:** protect the thing you depend on from too many requests.
*About 15 minutes. No coding needed to follow along.*

---

### The idea in plain words
Two ways to handle a crowd:

* **Rate limit = a turnstile.** People get through one at a time at a fixed speed. Extra people **wait** in line.
* **Backpressure = a full club.** When the club is full the doorman says "sorry, we are full" and **turns people away** rather than letting a dangerous crowd build up.

| | Rate limit | Backpressure |
| --- | --- | --- |
| Extra requests | Wait their turn | Are rejected ("503 busy") |
| Use when | Callers can tolerate a delay | A spike must not pile up |

### What went wrong in L2
In L2 the agent handled one request at a time, so the question never came up. Production has many customers at once. Without limits, the agent can knock over the very tool it needs.

### What we do now (L3)
We add a turnstile in front of the tool, and a "club capacity" for how many requests we accept at the same moment.

In [ ]:
# Step 0 - Connect to the AI model (Ollama). Run this cell first.
from langchain_ollama import ChatOllama

llm = ChatOllama(model="gpt-oss:120b-cloud", temperature=0)
try:
    print("Model says:", llm.invoke("Reply with just the word: ready").content)
except Exception as problem:
    print("The model is not reachable yet.")
    print("Fix: open a terminal, run   ollama signin   then restart this notebook.")
    print("Technical detail:", type(problem).__name__)

### Part A: the turnstile (rate limit)
Rule: **at most 4 calls per second**. A 5th call within the same second must wait.

The agent has a new box called `wait_turn` in front of the tool:
```
START -> [ wait_turn ] -> [ call_tool ] -> END
```

In [ ]:
import time
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

RATE_LIMIT = 4        # calls allowed ...
RATE_WINDOW = 1.0     # ... per this many seconds
recent_calls = []     # when did we last call the tool?

class State(TypedDict):
    n: int
    waited: float
    answer: str

def wait_turn(state):
    now = time.time()
    recent = [t for t in recent_calls if now - t < RATE_WINDOW]
    recent_calls[:] = recent
    waited = 0.0
    if len(recent) >= RATE_LIMIT:                         # the turnstile is full: wait for the oldest call to expire
        waited = RATE_WINDOW - (now - recent[0])
        time.sleep(waited)
    recent_calls.append(time.time())
    return {"waited": waited}

def call_tool(state):
    return {"answer": "balance 1,250 INR"}

builder = StateGraph(State)
builder.add_node("wait_turn", wait_turn)
builder.add_node("call_tool", call_tool)
builder.add_edge(START, "wait_turn")
builder.add_edge("wait_turn", "call_tool")
builder.add_edge("call_tool", END)
turnstile_agent = builder.compile()

Six customers ask **quickly one after another**. Predict which one has to wait.

In [ ]:
recent_calls.clear()
for n in range(1, 7):
    result = turnstile_agent.invoke({"n": n, "waited": 0.0, "answer": ""})
    note = f"  <- had to wait {result['waited']:.1f}s at the turnstile" if result["waited"] > 0 else ""
    print(f"customer {n}: {result['answer']}{note}")

**What you should see:** customers 1 to 4 go straight through. Customer 5 waits about a second. Nobody is turned away, they are just slowed down.

---
### Part B: the full club (backpressure)
Now the opposite choice. The tool can only serve **3 customers at the same moment**. Each call takes half a second. Six customers arrive **together**.

The new first box is `admit`. If the club is full it sends the customer to `busy` instead:
```
START -> [ admit ] --room--> [ call_tool ] -> END
                \--full----> [ busy ] -------> END
```

In [ ]:
import threading
from concurrent.futures import ThreadPoolExecutor

MAX_AT_ONCE = 3
seats = threading.BoundedSemaphore(MAX_AT_ONCE)     # three seats in the club

class ClubState(TypedDict):
    n: int
    admitted: bool
    answer: str

def admit(state):
    got_a_seat = seats.acquire(blocking=False)       # do not wait: either there is a seat or there is not
    return {"admitted": got_a_seat}

def serve(state):
    time.sleep(0.5)                                  # the tool is busy for half a second
    seats.release()                                  # leave the seat for the next customer
    return {"answer": "balance 1,250 INR"}

def busy(state):
    return {"answer": "503 busy, please try again shortly"}

club = StateGraph(ClubState)
club.add_node("admit", admit)
club.add_node("serve", serve)
club.add_node("busy", busy)
club.add_edge(START, "admit")
club.add_conditional_edges("admit", lambda s: "room" if s["admitted"] else "full", {"room": "serve", "full": "busy"})
club.add_edge("serve", END)
club.add_edge("busy", END)
club_agent = club.compile()

def arrive(n):
    return n, club_agent.invoke({"n": n, "admitted": False, "answer": ""})["answer"]

with ThreadPoolExecutor(max_workers=6) as crowd:       # six customers arrive at the same moment
    for n, answer in sorted(crowd.map(arrive, range(1, 7))):
        print(f"customer {n}: {answer}")

**What you should see:** three customers are served, the other three are told "503 busy". The tool was never asked to do more than it can handle. (Which three get seats can differ run to run, because they arrive at the same instant.)

---
### Your turn: predict first, then run
Change the club to **5 seats** and run Part B again.

**Predict:** how many customers are turned away?

In [ ]:
MAX_AT_ONCE = 5
seats = threading.BoundedSemaphore(MAX_AT_ONCE)
with ThreadPoolExecutor(max_workers=6) as crowd:
    for n, answer in sorted(crowd.map(arrive, range(1, 7))):
        print(f"customer {n}: {answer}")

**Answer:** one. Five seats are taken, the sixth customer is turned away.

---

### Remember
* **Rate limit:** make callers wait. **Backpressure:** turn extra work away. Both protect the tool.
* A fast, polite "busy" is much better than letting everything pile up and collapse.

> **Technical note (optional):** this demo uses threads to keep it small. Real concurrency, queues and scaling are covered on Day 17.

**Next notebook:** put everything together and rehearse the failures.